# 🏦 Australian Credit Approval Prediction
### Comparing Logistic Regression vs SVM with Cross-Validation, Feature Selection & ROC Analysis

[![Python](https://img.shields.io/badge/Python-3.10-blue)](https://python.org)
[![scikit-learn](https://img.shields.io/badge/scikit--learn-1.x-orange)](https://scikit-learn.org)
[![Dataset](https://img.shields.io/badge/Dataset-UCI%20Australian%20Credit-green)](https://archive.ics.uci.edu/dataset/143/statlog+australian+credit+approval)


## 📋 Project Overview

Credit scoring is one of the most impactful applications of machine learning in financial services. Lenders must balance **risk** (approving a bad applicant) against **opportunity cost** (rejecting a good one). A well-calibrated model helps automate and improve these decisions at scale.

This project builds an end-to-end classification pipeline on the **UCI Australian Credit Approval** dataset (Quinlan, 1987) to:
- Predict whether a credit card application should be **approved (+1)** or **rejected (0)**
- Compare two interpretable classifiers: **Logistic Regression** and **Support Vector Machine (SVM)**
- Use **stratified cross-validation**, **hyperparameter tuning (GridSearchCV)**, and **feature selection (SelectKBest)** for rigorous evaluation

**Dataset at a glance:**
| Property | Value |
|---|---|
| Instances | 690 |
| Features | 14 (all numerical after encoding) |
| Target classes | Approved (+1), Rejected (0) |
| Class balance | 44.5% approved / 55.5% rejected |

> ⚠️ Feature names are anonymised in the original dataset for confidentiality reasons.


## 1. Environment Setup

In [ ]:
# Standard data science stack
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: preprocessing, selection, models, evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RANDOM_STATE = 42
print("Libraries loaded successfully.")


## 2. Load Dataset

The dataset is loaded directly from the **UCI Machine Learning Repository** — no local file needed.
This makes the notebook fully reproducible for anyone who clones the repository.


In [ ]:
# Load directly from UCI — reproducible, no local path needed
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/australian/australian.dat"

col_names = [f"A{i}" for i in range(1, 15)] + ["Class"]

df = pd.read_csv(url, sep=" ", header=None, names=col_names)

print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── Shape & types ──────────────────────────────────────────────────────────────
print("Shape:", df.shape)
print()
print("Data types:")
print(df.dtypes)
print()

# ── Missing values ─────────────────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None — dataset is clean ✓")
print()

# ── Duplicate rows ─────────────────────────────────────────────────────────────
print(f"Duplicate rows: {df.duplicated().sum()}")
print()

# ── Summary statistics ─────────────────────────────────────────────────────────
df.describe().round(2)


In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df["Class"].value_counts().sort_index()
axes[0].bar(["Rejected (0)", "Approved (1)"], counts.values,
            color=["#e74c3c", "#2ecc71"], edgecolor="white", linewidth=1.5)
axes[0].set_title("Class Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, f"{v}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=11)

# Selected numerical feature distributions
numerical_preview = ["A2", "A3", "A8", "A11", "A14", "A15"]
df[numerical_preview].plot(kind="box", ax=axes[1], vert=False, patch_artist=True)
axes[1].set_title("Selected Feature Distributions (Boxplot)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Value")

plt.tight_layout()
plt.show()


In [ ]:
# ── Correlation analysis ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap="coolwarm",
            linewidths=0.5, ax=axes[0])
axes[0].set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")

# Correlation with target
corr_with_target = corr_matrix["Class"].drop("Class").sort_values()
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr_with_target]
axes[1].barh(corr_with_target.index, corr_with_target.values, color=colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Feature Correlation with Target (Class)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Pearson Correlation")

plt.tight_layout()
plt.show()

print("Top 5 features positively correlated with approval:")
print(corr_with_target.tail(5).to_string())


**Key EDA findings:**
- The dataset is slightly imbalanced (55.5% rejected), but not severely — accuracy remains a valid metric
- Features **A8, A9, A10** show the strongest positive correlation with approval
- **A3, A15** show some right-skewed distributions (outliers present), handled via StandardScaler
- No missing values or duplicates — the dataset is clean


## 4. Preprocessing

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

# ── Train / test split (80 / 20, stratified) ──────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")

# ── Feature scaling (StandardScaler fit on train only) ────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)       # ← avoids data leakage

# ── Feature selection (SelectKBest, k=10) ────────────────────────────────────
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_train_scaled, y_train)

X_train_sel = selector.transform(X_train_scaled)
X_test_sel  = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()
print(f"\nSelected features (top 10 by ANOVA F-score): {selected_features}")


## 5. Model Training & Hyperparameter Tuning

Both models are tuned using **5-fold Stratified Cross-Validation** with `GridSearchCV`, scoring on **ROC-AUC** to account for the class imbalance.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── Logistic Regression ────────────────────────────────────────────────────────
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid={"C": [0.01, 0.1, 1, 10]},
    cv=skf, scoring="roc_auc", n_jobs=-1
)
lr_grid.fit(X_train_sel, y_train)
best_lr = lr_grid.best_estimator_
print(f"Logistic Regression — best C: {lr_grid.best_params_['C']}  |  CV AUC: {lr_grid.best_score_:.4f}")

# ── SVM ────────────────────────────────────────────────────────────────────────
svm_grid = GridSearchCV(
    SVC(probability=True, random_state=RANDOM_STATE),
    param_grid={"C": [0.1, 1, 10], "kernel": ["linear", "rbf"]},
    cv=skf, scoring="roc_auc", n_jobs=-1
)
svm_grid.fit(X_train_sel, y_train)
best_svm = svm_grid.best_estimator_
print(f"SVM              — best params: {svm_grid.best_params_}  |  CV AUC: {svm_grid.best_score_:.4f}")


## 6. Evaluation on Held-Out Test Set

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_lr  = best_lr.predict(X_test_sel)
y_prob_lr  = best_lr.predict_proba(X_test_sel)[:, 1]

y_pred_svm = best_svm.predict(X_test_sel)
y_prob_svm = best_svm.predict_proba(X_test_sel)[:, 1]

# ── Summary table ─────────────────────────────────────────────────────────────
results = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM"],
    "Accuracy":  [accuracy_score(y_test, y_pred_lr),  accuracy_score(y_test, y_pred_svm)],
    "ROC-AUC":   [roc_auc_score(y_test, y_prob_lr),   roc_auc_score(y_test, y_prob_svm)],
}).set_index("Model").round(4)

print("=" * 40)
print(results.to_string())
print("=" * 40)


In [ ]:
# ── Detailed classification reports ──────────────────────────────────────────
print("LOGISTIC REGRESSION")
print(classification_report(y_test, y_pred_lr, target_names=["Rejected", "Approved"]))

print("SVM")
print(classification_report(y_test, y_pred_svm, target_names=["Rejected", "Approved"]))


In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, title in zip(axes,
                             [y_pred_lr, y_pred_svm],
                             ["Logistic Regression", "SVM"]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, preds),
                           display_labels=["Rejected", "Approved"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(title, fontsize=13, fontweight="bold")

plt.suptitle("Confusion Matrices — Test Set", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# ── ROC curves ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

for probs, label, color in [
    (y_prob_lr,  f"Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})",  "#3498db"),
    (y_prob_svm, f"SVM               (AUC = {roc_auc_score(y_test, y_prob_svm):.3f})", "#e74c3c"),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax.plot(fpr, tpr, label=label, linewidth=2, color=color)

ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random classifier")
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Sensitivity / Recall)", fontsize=11)
ax.set_title("ROC Curve Comparison", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()


## 7. Conclusion

### Results Summary

| Model | Accuracy | ROC-AUC |
|---|---|---|
| Logistic Regression | **87.68%** | 0.9115 |
| SVM (RBF kernel) | 86.23% | **0.9166** |

### Business Interpretation

Both models perform well, but they serve different business needs:

- **Logistic Regression** is the preferred choice when **interpretability matters** — a compliance officer or risk manager can audit the model's coefficients and understand *why* a specific application was approved or rejected. Its higher accuracy (87.68%) means fewer misclassifications overall.

- **SVM** achieves a marginally better **ROC-AUC (0.9166)**, meaning it has slightly stronger discrimination across all decision thresholds. This is valuable when the business wants to tune its approval cutoff (e.g., tightening or loosening risk appetite) rather than using a fixed 50% threshold.

In a real deployment, the choice of model and threshold would be guided by the **cost asymmetry** between the two error types:
- A **false positive** (approving a bad applicant) carries credit loss risk
- A **false negative** (rejecting a good applicant) means lost revenue

### What I Would Do Next
- Add tree-based models (Random Forest, XGBoost) for comparison
- Incorporate threshold optimisation based on a business cost function
- Explore SHAP values for explainability
